In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from plotting import *
setup_plotting_style()


In [ ]:
import pandas as pd
import numpy as np
import joblib
import os


In [ ]:
# Load the three datasets
print("Loading datasets...")
try:
    train_df = pd.read_csv(MACHINE_A_TRAIN_SET_FILE)
    val_df   = pd.read_csv(MACHINE_A_VAL_SET_FILE)
    test_df  = pd.read_csv(MACHINE_A_TEST_SET_FILE)
    
    X_train = train_df[MACHINE_A_FEATURES]
    y_train = train_df['label']

    X_val   = val_df[MACHINE_A_FEATURES]
    y_val   = val_df['label']

    X_test  = test_df[MACHINE_A_FEATURES]
    y_test  = test_df['label']

        
    print(f"Loaded successfully.")
    print(f"Train: {train_df.shape}")
    print(f"Val:   {val_df.shape}")
    print(f"Test:  {test_df.shape}")

except FileNotFoundError as e:
    print(f"ERROR: Could not find split files. Did you run Notebook 22? \n{e}")


### Custom StandardScaler Implementation


In [ ]:
class CustomStandardScaler:
    """
    Standardizes features by removing the mean and scaling to unit variance (z-score normalization).
    Why normalize? Features with larger scales (e.g., thousands vs decimals) can dominate the 
    gradient descent process, causing it to converge slowly or find suboptimal solutions. 
    By centering around 0 and scaling the standard deviation to 1, gradient descent updates 
    weights more uniformly.
    """
    def __init__(self):
        self.feature_means = None
        self.feature_std_devs = None

    def fit(self, features_data):
        features_array = np.array(features_data)
        
        # Calculate the mean for each feature across all samples
        self.feature_means = np.mean(features_array, axis=0)
        
        # Calculate the standard deviation for each feature
        self.feature_std_devs = np.std(features_array, axis=0)
        
        # Avoid division by zero for constant features (where standard deviation is 0)
        self.feature_std_devs[self.feature_std_devs == 0] = 1.0
        return self

    def transform(self, features_data):
        features_array = np.array(features_data)
        
        # Apply the z-score formula: z = (x - mean) / standard_deviation
        return (features_array - self.feature_means) / self.feature_std_devs

    def fit_transform(self, features_data):
        return self.fit(features_data).transform(features_data)


### Custom Logistic Regression Implementation


In [ ]:
class CustomLogisticRegression:
    def __init__(self, inverse_regularization_strength=1.0, learning_rate=0.1, max_iterations=2000, class_weight_strategy=None, random_state=42):
        # 'inverse_regularization_strength' determines the penalty for large weights.
        # Smaller values specify stronger regularization.
        self.inverse_regularization_strength = inverse_regularization_strength
        
        # The step size for each iteration of gradient descent.
        self.learning_rate = learning_rate
        
        # Maximum number of passes over the training data
        self.max_iterations = max_iterations
        
        # Strategy to handle class imbalance (e.g. 'balanced')
        self.class_weight_strategy = class_weight_strategy
        self.random_state = random_state
        
        # Model parameters that will be learned
        self.feature_weights = None
        self.bias_term = None

    def _sigmoid(self, log_odds):
        """
        The Sigmoid Function squashes raw linear outputs (log-odds) into a valid probability range [0, 1].
        Equation: f(x) = 1 / (1 + e^-x)
        """
        # We clip the log-odds values to prevent numerical overflow in np.exp (which would cause infinite/NaN results)
        clipped_log_odds = np.clip(log_odds, -250, 250)
        return 1 / (1 + np.exp(-clipped_log_odds))

    def fit(self, feature_matrix, labels):
        np.random.seed(self.random_state)
        number_of_samples, number_of_features = feature_matrix.shape
        
        # Initialize weights to zeros. Gradient descent will adjust these iteratively.
        self.feature_weights = np.zeros(number_of_features)
        self.bias_term = 0

        # L2 Regularization (Ridge) penalty term (lambda).
        # L2 adds a penalty for large weights to the loss function to prevent overfitting.
        # We define lambda as the inverse of the given regularization strength.
        lambda_penalty = 1 / self.inverse_regularization_strength if self.inverse_regularization_strength != 0 else 0

        # Calculate class weights if the 'balanced' strategy is requested to handle imbalanced datasets.
        weight_class_0, weight_class_1 = 1.0, 1.0
        if self.class_weight_strategy == 'balanced':
            count_class_0 = np.sum(labels == 0)
            count_class_1 = np.sum(labels == 1)
            total_samples = len(labels)
            
            # The formula increases the weight for the minority class.
            weight_class_0 = total_samples / (2.0 * count_class_0) if count_class_0 > 0 else 1.0
            weight_class_1 = total_samples / (2.0 * count_class_1) if count_class_1 > 0 else 1.0

        # Assign a weight to each sample based on its true label
        sample_weights = np.where(labels == 1, weight_class_1, weight_class_0)

        # Gradient Descent Loop
        for _ in range(self.max_iterations):
            # Forward Pass: Calculate the linear combination (z = X * w + b)
            linear_combination = np.dot(feature_matrix, self.feature_weights) + self.bias_term
            
            # Apply the sigmoid function to get the predicted probability (y_hat = sigmoid(z))
            predicted_probabilities = self._sigmoid(linear_combination)

            # Error Calculation: Difference between predicted probabilities and actual labels
            prediction_errors = predicted_probabilities - labels
            
            # Apply sample weights to the errors (gives more importance to minority class errors)
            weighted_prediction_errors = prediction_errors * sample_weights
            
            # --- Gradient Calculation ---
            # The gradient tells us the direction of steepest ascent of the loss function.
            # We calculate the dot product of the transposed feature matrix and the errors.
            # The L2 penalty derivative (lambda_penalty * weights / n_samples) is added here to shrink large weights.
            weight_gradients = (1 / number_of_samples) * np.dot(feature_matrix.T, weighted_prediction_errors) + (lambda_penalty / number_of_samples) * self.feature_weights
            
            # The bias gradient is simply the average of the weighted errors.
            bias_gradient = (1 / number_of_samples) * np.sum(weighted_prediction_errors)

            # --- Update Rule ---
            # We update the weights by taking a step in the *opposite* direction of the gradient 
            # (subtracting it), scaled by the learning_rate, to minimize the loss.
            self.feature_weights -= self.learning_rate * weight_gradients
            self.bias_term -= self.learning_rate * bias_gradient
            
        return self

    def predict_proba(self, feature_matrix):
        linear_combination = np.dot(feature_matrix, self.feature_weights) + self.bias_term
        probability_class_1 = self._sigmoid(linear_combination)
        probability_class_0 = 1 - probability_class_1
        
        # Return probabilities for both classes: [P(y=0), P(y=1)]
        return np.vstack([probability_class_0, probability_class_1]).T

    def predict(self, feature_matrix, decision_threshold=0.5):
        # Predict class 1 if its probability is >= the threshold, else predict class 0
        probabilities = self.predict_proba(feature_matrix)
        return (probabilities[:, 1] >= decision_threshold).astype(int)


### Custom Metrics


In [ ]:
def custom_confusion_matrix(actual_labels, predicted_labels):
    actual_labels = np.asarray(actual_labels)
    predicted_labels = np.asarray(predicted_labels)
    
    true_positives = np.sum((actual_labels == 1) & (predicted_labels == 1))
    true_negatives = np.sum((actual_labels == 0) & (predicted_labels == 0))
    false_positives = np.sum((actual_labels == 0) & (predicted_labels == 1))
    false_negatives = np.sum((actual_labels == 1) & (predicted_labels == 0))
    
    return np.array([[true_negatives, false_positives], [false_negatives, true_positives]])

def custom_f1_score(actual_labels, predicted_labels):
    confusion_mat = custom_confusion_matrix(actual_labels, predicted_labels)
    true_negatives, false_positives = confusion_mat[0]
    false_negatives, true_positives = confusion_mat[1]
    
    # Precision: Of all positive predictions, how many were correct?
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    
    # Recall: Of all actual positives, how many did we correctly predict?
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    
    # F1 Score is the harmonic mean of precision and recall
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)


In [ ]:
# Scale the data
scaler = CustomStandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Hyperparameter search: Values for Inverse Regularization Strength (C)
inv_reg_strength_candidates = [0.001, 0.01, 0.1, 1, 10, 100]

best_f1_score = 0
best_trained_model = None
best_inv_reg_strength = None
evaluation_history = []

print(f"{'Inv Reg Strength':<20} | {'Val F1 Score':<15} | {'Val Accuracy':<15}")
print("-" * 55)

for inv_reg_strength in inv_reg_strength_candidates:
    # Train
    model = CustomLogisticRegression(
        inverse_regularization_strength=inv_reg_strength, 
        max_iterations=2000, 
        learning_rate=0.1, 
        class_weight_strategy='balanced'
    )
    model.fit(X_train_scaled, np.array(y_train))
    
    # Validate
    validation_predictions = model.predict(X_val_scaled)
    validation_f1 = custom_f1_score(np.array(y_val), validation_predictions)
    validation_accuracy = np.mean(validation_predictions == np.array(y_val))

    evaluation_history.append({'Inv_Reg_Strength': inv_reg_strength, 'F1': validation_f1})
    print(f"{inv_reg_strength:<20} | {validation_f1:.5f}         | {validation_accuracy:.5f}")

    # Track Best Model based on Validation F1 Score
    if validation_f1 > best_f1_score:
        best_f1_score = validation_f1
        import copy
        best_trained_model = copy.deepcopy(model)
        best_inv_reg_strength = inv_reg_strength

print("-" * 55)
print(f"Winner: Inverse Regularization Strength={best_inv_reg_strength} with F1={best_f1_score:.5f}")


In [ ]:
# Final Test & Save

# Final Prediction on Test Set using the best performing model
test_predictions = best_trained_model.predict(X_test_scaled)

print("=== Final Test Set Report ===")
final_confusion_matrix = custom_confusion_matrix(np.array(y_test), test_predictions)
print("Confusion Matrix:")
print(final_confusion_matrix)

test_f1 = custom_f1_score(np.array(y_test), test_predictions)
test_accuracy = np.mean(test_predictions == np.array(y_test))
print(f"\nTest F1 Score: {test_f1:.5f}")
print(f"Test Accuracy: {test_accuracy:.5f}")

# Confusion Matrix Plot
plt.figure(figsize=(6, 5))
sns.heatmap(final_confusion_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Clean', 'Pred Jamming'],
            yticklabels=['True Clean', 'True Jamming'])
plt.title('Confusion Matrix (Test Set)')
plt.show()

# Bundle scaler and model into a dict to save them together
custom_pipeline = {
    'scaler': scaler,
    'model': best_trained_model
}

# Save using a custom filename to avoid overriding sklearn pipeline
custom_model_file = str(MACHINE_A_MODEL_FILE).replace('.pkl', '_custom.pkl')
joblib.dump(custom_pipeline, custom_model_file)

print(f"Model successfully saved to: {custom_model_file}")
